# 01 — Data Cleaning Pipeline

This notebook demonstrates the cleaning pipeline from `src/data/cleaner.py`.

**Pipeline steps:**
1. Load raw dataset (4,801 rows)
2. Deduplicate (exact + near-duplicates by lat/lon)
3. Borough-aware median imputation (BEDS, BATH)
4. Outlier capping (IQR x 3 — cap, don't drop)
5. Text normalization (lowercase, strip)
6. Borough / ZIP / type standardization
7. Validation (no nulls, positive PRICE/SQFT, NYC lat/lon range)

All logic lives in `src/data/cleaner.py` — this notebook is a thin orchestrator.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data.loader import load_raw, load_cleaned
from src.data.cleaner import clean_pipeline
from src.utils.validation import validate_cleaned_data
from src.utils.logging_config import setup_logging

setup_logging()

## Load raw data

In [ ]:
df_raw = load_raw()
print(f"Raw: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
df_raw.head(3)

## Run cleaning pipeline

In [ ]:
df_clean = clean_pipeline(df_raw)
print(f"Cleaned: {df_clean.shape[0]} rows x {df_clean.shape[1]} cols")
print(f"Dropped: {df_raw.shape[0] - df_clean.shape[0]} rows")
df_clean.describe()

## Validate cleaned data

In [ ]:
issues = validate_cleaned_data(df_clean)
if issues:
    print("VALIDATION ISSUES:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All validation checks passed")
    print(f"  Nulls: {df_clean.isnull().sum().sum()}")
    print(f"  Price range: ${df_clean['PRICE'].min():,.0f} - ${df_clean['PRICE'].max():,.0f}")
    print(f"  Boroughs: {df_clean['BOROUGH'].nunique() if 'BOROUGH' in df_clean.columns else 'N/A'}")